In [ ]:
import pandas as pd
import numpy as np


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving air_pollution_model_data.csv to air_pollution_model_data (2).csv
Saving combined_air_noise_data.csv to combined_air_noise_data (2).csv
Saving noise_pollution_model_data.csv to noise_pollution_model_data (2).csv


In [ ]:
df = pd.read_csv("combined_air_noise_data.csv")
df.shape


(2796, 17)

In [ ]:
df = df.sort_values(by=['City', 'Year', 'Month']).reset_index(drop=True)


In [ ]:
lag_columns = [
    'avg_AQI',
    'avg_PM25',
    'avg_PM10',
    'avg_NO2',
    'Day',
    'Night'
]

for col in lag_columns:
    for lag in [1, 2, 3]:
        df[f'{col}_lag{lag}'] = (
            df.groupby('City')[col].shift(lag)
        )


In [ ]:
rolling_cols = ['avg_AQI', 'avg_PM25', 'Day']

for col in rolling_cols:
    df[f'{col}_roll3'] = (
        df.groupby('City')[col]
        .rolling(window=3)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df[f'{col}_roll6'] = (
        df.groupby('City')[col]
        .rolling(window=6)
        .mean()
        .reset_index(level=0, drop=True)
    )


In [ ]:
df = df.dropna().reset_index(drop=True)
print(df.shape)


(2286, 41)


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['City_encoded'] = le.fit_transform(df['City'])


In [ ]:
df[['City', 'City_encoded']].head()


,City,City_encoded
0,Bengaluru,0
1,Bengaluru,0
2,Bengaluru,0
3,Bengaluru,0
4,Bengaluru,0


In [ ]:
df = df.drop(columns=['City'])


In [ ]:
air_target = 'avg_AQI'

air_features = [
    'Year', 'Month', 'City_encoded',
    'avg_PM25', 'avg_PM10', 'avg_NO2', 'avg_SO2', 'avg_CO', 'avg_O3',

    # Lag features
    'avg_AQI_lag1', 'avg_AQI_lag2', 'avg_AQI_lag3',
    'avg_PM25_lag1', 'avg_PM25_lag2', 'avg_PM25_lag3',
    'avg_PM10_lag1', 'avg_PM10_lag2',

    # Rolling features
    'avg_AQI_roll3', 'avg_AQI_roll6',
    'avg_PM25_roll3', 'avg_PM25_roll6'
]

air_df = df[air_features + [air_target]]
print(air_df.shape)
air_df.head()


(2286, 22)


,Year,Month,City_encoded,avg_PM25,avg_PM10,avg_NO2,avg_SO2,avg_CO,avg_O3,avg_AQI_lag1,...,avg_PM25_lag1,avg_PM25_lag2,avg_PM25_lag3,avg_PM10_lag1,avg_PM10_lag2,avg_AQI_roll3,avg_AQI_roll6,avg_PM25_roll3,avg_PM25_roll6,avg_AQI
0,2015,1,0,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,...,30.07,30.07,30.07,73.44,73.44,83.0,83.0,30.07,30.07,83.0
1,2015,1,0,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,...,30.07,30.07,30.07,73.44,73.44,83.0,83.0,30.07,30.07,83.0
2,2015,1,0,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,...,30.07,30.07,30.07,73.44,73.44,83.0,83.0,30.07,30.07,83.0
3,2015,1,0,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,...,30.07,30.07,30.07,73.44,73.44,83.0,83.0,30.07,30.07,83.0
4,2015,1,0,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,...,30.07,30.07,30.07,73.44,73.44,83.0,83.0,30.07,30.07,83.0


In [ ]:
noise_targets = ['Day', 'Night']

noise_features = [
    'Year', 'Month', 'City_encoded',
    'avg_PM25', 'avg_PM10', 'avg_NO2', 'avg_AQI',

    # Lag features
    'Day_lag1', 'Day_lag2', 'Day_lag3',
    'Night_lag1', 'Night_lag2',
    'avg_AQI_lag1',

    # Rolling features
    'Day_roll3', 'Day_roll6'
]

noise_df = df[noise_features + noise_targets]
print(noise_df.shape)
noise_df.head()


(2286, 17)


,Year,Month,City_encoded,avg_PM25,avg_PM10,avg_NO2,avg_AQI,Day_lag1,Day_lag2,Day_lag3,Night_lag1,Night_lag2,avg_AQI_lag1,Day_roll3,Day_roll6,Day,Night
0,2015,1,0,30.07,73.44,19.537419,83.0,58.0,65.0,54.0,56.0,56.0,83.0,65.000000,62.333333,72.0,63.0
1,2015,1,0,30.07,73.44,19.537419,83.0,72.0,58.0,65.0,63.0,56.0,83.0,63.000000,62.333333,59.0,55.0
2,2015,1,0,30.07,73.44,19.537419,83.0,59.0,72.0,58.0,55.0,63.0,83.0,66.000000,62.500000,67.0,61.0
3,2015,1,0,30.07,73.44,19.537419,83.0,67.0,59.0,72.0,61.0,55.0,83.0,61.666667,63.333333,59.0,57.0
4,2015,1,0,30.07,73.44,19.537419,83.0,59.0,67.0,59.0,57.0,61.0,83.0,63.000000,63.000000,63.0,60.0


In [ ]:
air_df.to_csv("air_pollution_final_ml.csv", index=False)
noise_df.to_csv("noise_pollution_final_ml.csv", index=False)
df.to_csv("combined_final_with_lags.csv", index=False)


In [ ]:
files.download("air_pollution_final_ml.csv")
files.download("noise_pollution_final_ml.csv")
files.download("combined_final_with_lags.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>